# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjds22/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — Search visibility is uneven.** The March 2026 data contained 3,611,061 observations. Median impressions were 16, compared with a mean of 77.72, indicating a strongly right-skewed distribution. This supports investigating low-visibility observations, but does not establish that these observations need content changes.

**Finding 2 — The decision tree outperformed the baseline on the held-out sample.** The decision tree achieved an F1 score of 0.847 compared with 0.775 for the rule-based baseline. However, the evaluation used a sampled time-based split and a next-month low-impressions target. The result supports further testing, not a claim that the model improves search rankings or that content changes cause better performance.

**Methodology questions:** Are the results representative of all content or only the sampled observations? Would the comparison remain similar on a later time period and on previously unseen clients? Are missing observations systematically different from observed ones? These questions limit how broadly the findings should be interpreted.

In [ ]:

import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

audit = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {fact_daily}
WHERE month = '2026-03'
  AND gsc_data_available = TRUE
""").df()

print("March 2026 distribution check:")
display(audit.describe(percentiles=[0.5, 0.75, 0.95]).round(2))


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I compare a random holdout with a time-aware holdout to assess sensitivity to the validation design. The target is whether a content observation has fewer than 100 GSC impressions in the following month. The time-aware test uses May 2026 features to predict June 2026 outcomes, with training restricted to earlier feature months. Both methods use the same features, decision-tree settings and evaluation metrics. Random splitting may give an overly optimistic estimate when observations from the same content appear in both training and test data, so the time-aware result is more relevant to forecasting future observations. Neither split establishes causation.

In [ ]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

# Aggregate daily observations into monthly content-level records.
monthly = con.sql(f"""
    SELECT
        month,
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM {fact_daily}
    WHERE month >= '2025-01'
      AND month <= '2026-06'
      AND gsc_data_available = TRUE
    GROUP BY month, client_hash_id, content_hash_id
""").df()

monthly["month_date"] = pd.to_datetime(
    monthly["month"].astype(str), format="%Y-%m", errors="coerce"
)
monthly = monthly.dropna(subset=["month_date"])

# Attach next-month impressions as the outcome.
future = monthly[
    ["client_hash_id", "content_hash_id", "month_date", "impressions"]
].copy()
future["month_date"] = future["month_date"] - pd.DateOffset(months=1)
future = future.rename(columns={"impressions": "next_impressions"})

data = monthly.merge(
    future,
    on=["client_hash_id", "content_hash_id", "month_date"],
    how="inner"
)

data["target"] = (data["next_impressions"] < 100).astype(int)
features = ["impressions", "clicks", "avg_position"]

data = data.replace([np.inf, -np.inf], np.nan).dropna(
    subset=features + ["target"]
)

# Random split: a comparison that may benefit from repeated content.
eligible = data[data["month_date"] <= "2026-05-01"].copy()
random_sample = eligible.sample(
    n=min(180000, len(eligible)), random_state=42
)

random_train, random_test = train_test_split(
    random_sample,
    test_size=0.2,
    random_state=42,
    stratify=random_sample["target"]
)

# Time-aware split: earlier months train; May predicts June.
time_train = data[data["month_date"] <= "2026-04-01"].copy()
time_test = data[data["month_date"] == "2026-05-01"].copy()

time_train = time_train.sample(
    n=min(150000, len(time_train)), random_state=42
)
time_test = time_test.sample(
    n=min(30000, len(time_test)), random_state=42
)

def evaluate_split(train_df, test_df, split_name):
    model_check = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=100,
        random_state=42,
        class_weight="balanced"
    )
    model_check.fit(train_df[features], train_df["target"])
    pred = model_check.predict(test_df[features])
    y = test_df["target"]

    return {
        "Split": split_name,
        "Train rows": len(train_df),
        "Test rows": len(test_df),
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0)
    }

validation_results = pd.DataFrame([
    evaluate_split(random_train, random_test, "Random split"),
    evaluate_split(time_train, time_test, "Time-aware split")
])

display(validation_results.round(3))


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The model uses current-month impressions, clicks and average position to predict whether impressions will be below 100 in the following month. The next-month impressions field is used to create the target and must not be included among the model features. The time-aware evaluation trains on feature months through April 2026 and tests on May 2026 features against June 2026 outcomes. This avoids training on later feature months, but does not rule out every source of bias, including repeated content, missing observations or changes in the data-generating process.

In [ ]:

# Audit the final feature list and time split.
forbidden_features = {
    "next_impressions",
    "target",
    "report_date",
    "month_date",
    "future_label"
}

print("Final model features:", features)
print("Forbidden outcome/time fields used as features:",
      sorted(set(features) & forbidden_features))

print("Target is defined from next-month impressions:",
      "next_impressions" in data.columns)

print("Time-aware training ends:",
      time_train["month_date"].max().date())

print("Time-aware test feature month:",
      time_test["month_date"].min().strftime("%Y-%m"))

print("Time-aware test feature months:",
      sorted(time_test["month_date"].dt.strftime("%Y-%m").unique()))

assert not (set(features) & forbidden_features), \
    "Leakage risk: an outcome or time field is in the feature list."

assert time_train["month_date"].max() < time_test["month_date"].min(), \
    "Leakage risk: training includes the test feature month."

print("\nChecks passed for the listed features and temporal ordering.")


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim:** The decision tree is better at identifying content that will have low search impressions.

**Rewritten claim:** In the sampled time-aware evaluation, the decision tree achieved a higher F1 score than the Week 4 rule-based baseline for predicting next-month low impressions. This is an observed result for the chosen sample, target and split. Further validation is needed to assess generalisation to other time periods and clients. The model provides directional decision support and does not prove that changing content will improve search performance.

In [ ]:

# Record the claim's scope using the validation results above.
print("Claim audit")
print("- Outcome: next-month impressions below 100")
print("- Features: current-month impressions, clicks, average position")
print("- Evaluation: random and time-aware holdouts")
print("- Interpretation: predictive association, not causation")
print("- Required caution: report the actual split metrics and sample sizes")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.